# Notebook D2: does the block prior also move the mouse's *lapses*? (CPU only, a few minutes)

**Question (tracker q2c).** Even at 100% contrast, where the stimulus is unmistakable, mice make a few errors (*lapses*). Do these errors depend on the block? If the animal sometimes ignores the stimulus and just guesses, it might guess toward the side the block favors. In the data from notebook D, accuracy at 100% contrast is about 0.93 when the stimulus is on the block-favored side (congruent) and 0.87 when it is on the other side (conflict). A shift of the decision boundary alone cannot produce that at full contrast, so the lapses may carry the prior too.

**Model** (per mouse; blocks 0.2/0.8 only):
$$P(\text{right}) = \lambda_{lo} + (1-\lambda_{lo}-\lambda_{hi})\,\sigma\!\big(b+\beta s+\gamma x\big),\qquad \lambda_{lo,hi}=0.49\,\sigma(a_{lo,hi}+d_{lo,hi}\,u)$$
- $x$ is the prior regressor in the logit (the block log-odds $\pm1.386$, or the recent-10-trial log-odds that fitted these mice better in notebook D) and $u$ is the same quantity scaled to about $\pm1$.
- **A0 / B0:** constant lapses ($d_L=d_R=0$). **A1 / B1:** lapses depend on the prior ($d_L, d_R$ free; 2 extra parameters).
- $\lambda_{lo}$ = probability of choosing *right* when the stimulus is strongly left (the error rate on left stimuli); $\lambda_{hi}$ = probability of choosing *left* when the stimulus is strongly right (the error rate on right stimuli).
- Lapses that follow the prior mean: in a right block, *more* errors on left stimuli ($d_{lo}>0$) and *fewer* errors on right stimuli ($d_{hi}<0$).
- Model-free check first: errors at 100% contrast by stimulus side and block.

**Needs one file:** `ibl_trials_tidy.csv` from notebook D's output (`D_out.zip`). Upload it as a Kaggle dataset and attach it, or place it next to this notebook. Nothing else is needed (no GPU, no Internet, no login).

In [ ]:
import os, glob, math, time, warnings
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
OUT = "/kaggle/working/D2_out" if os.path.exists("/kaggle/working") else "D2_out"; os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)
paths = glob.glob("/kaggle/input/**/ibl_trials_tidy.csv", recursive=True) + glob.glob("ibl_trials_tidy.csv") + glob.glob("/kaggle/working/**/ibl_trials_tidy.csv", recursive=True)
assert paths, "ibl_trials_tidy.csv not found: upload it (from D_out.zip) as a Kaggle dataset and attach it, or put it next to this notebook"
T = pd.read_csv(paths[0]).sort_values(["eid", "trial"]).reset_index(drop=True); print("using", paths[0], T.shape)
def recent_logodds(g, w=10):
    r = g.right.astype(float).values; out = np.full(len(r), np.nan)
    for i in range(w, len(r)): f = (r[i - w:i].sum() + .5) / (w + 1); out[i] = np.log(f / (1 - f))
    return pd.Series(out, index=g.index)
T["tp"] = T.groupby("eid", group_keys=False).apply(recent_logodds)
acc1 = T[T.ok & np.isclose(T.contrast, 1.0)].groupby("eid").apply(lambda d: (d.chose_right == d.right.astype(int)).mean())
B = T[T.ok & T.p_right.isin([0.2, 0.8]) & T.tp.notna() & ~np.isclose(T.contrast, 0.5)].copy()       # contrast 0.5 rows are from a different task variant (see notebook D)
cnt = B.groupby("subject").size(); B = B[B.subject.isin(cnt[cnt >= 600].index)].copy()
B["pi"] = np.log(B.p_right / (1 - B.p_right)); B["acc"] = (B.chose_right == B.right.astype(int)); B["fav"] = ((B.p_right == .8) & B.right) | ((B.p_right == .2) & ~B.right)
say(f"mice {B.subject.nunique()}, trials {len(B)}; sessions with accuracy < 0.8 at 100% contrast: {(acc1 < 0.8).sum()} of {len(acc1)}")

In [ ]:
# ===================== model-free: errors at high contrast by stimulus side and block =====================
say("\n[D2-1] accuracy by contrast: stimulus on the block-FAVORED side (congruent) vs the other side (conflict)")
rows = []
for c in sorted(B.contrast.unique()):
    x = B[np.isclose(B.contrast, c)]
    if c == 0: continue
    rows.append(dict(contrast=c, congruent=x[x.fav].acc.mean(), conflict=x[~x.fav].acc.mean(), n_cong=int(x.fav.sum()), n_conf=int((~x.fav).sum())))
say(pd.DataFrame(rows).round(3).to_string(index=False))
rb = np.random.default_rng(0)
def per_mouse_gap(c):
    x = B[np.isclose(B.contrast, c)]; g = x.groupby(["subject", "fav"]).acc.mean().unstack().dropna(); return (g[True] - g[False])
say("\nper-mouse gap in accuracy (congruent minus conflict) at high contrast; CI = bootstrap over mice; sign test over mice")
for c in (0.25, 1.0):
    g = per_mouse_gap(c).values; bs = [np.median(g[rb.integers(0, len(g), len(g))]) for _ in range(2000)]
    say(f"  contrast {c}: median gap {np.median(g):+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}] | mice with gap > 0: {int((g > 0).sum())}/{len(g)} (sign test p = {stats.binomtest(int((g > 0).sum()), len(g), .5).pvalue:.4f})")
say("\nerror rate at 100% contrast by stimulus side and block (pooled, all mice)")
x = B[np.isclose(B.contrast, 1.0)]; tab = x.assign(side=np.where(x.right, "stim right", "stim left"), block=np.where(x.p_right == .8, "right block (0.8)", "left block (0.2)")).groupby(["side", "block"]).agg(error=("acc", lambda a: 1 - a.mean()), n=("acc", "size")).round(3)
say(tab.to_string())

In [ ]:
# ===================== models =====================
def sig(x): return 1 / (1 + np.exp(-np.clip(x, -30, 30)))
CAP = 0.49
def nll_fn(d, which, lapse_dep):
    s, y = d.s.values, d.chose_right.values.astype(float)
    x = d.pi.values if which == "A" else d.tp.values; u = x / 1.386
    def f(p):
        lin = p[0] + p[1] * s + p[2] * x
        lL = CAP * sig(p[3] + p[5] * u); lR = CAP * sig(p[4] + p[6] * u)
        pr = np.clip(lL + (1 - lL - lR) * sig(lin), 1e-6, 1 - 1e-6); return -(y * np.log(pr) + (1 - y) * np.log(1 - pr)).sum()
    return f
BND = [(-3, 3), (0, 60), (-3, 3), (-8, 2), (-8, 2), (-4, 4), (-4, 4)]
def fit(d, which, lapse_dep, starts=5, seed=0):
    f = nll_fn(d, which, lapse_dep); rng = np.random.default_rng(seed); bnd = list(BND)
    if not lapse_dep: bnd[5] = bnd[6] = (0, 1e-9)
    best = None
    for _ in range(starts):
        x0 = [rng.normal(0, .1), rng.uniform(5, 15), rng.uniform(0, .4), rng.uniform(-4, -2.5), rng.uniform(-4, -2.5), 0.0 if not lapse_dep else rng.normal(0, .2), 0.0 if not lapse_dep else rng.normal(0, .2)]
        r = minimize(f, x0, method="L-BFGS-B", bounds=bnd)
        if best is None or r.fun < best.fun: best = r
    return best
def lapses(p, u): return CAP * sig(p[3] + p[5] * u), CAP * sig(p[4] + p[6] * u)
def run(mice, label):
    rows = []; t0 = time.time()
    for k, (sj, d) in enumerate(B[B.subject.isin(mice)].groupby("subject")):
        rec = dict(subject=sj, n=len(d))
        for which in ("A", "B"):
            f0, f1 = fit(d, which, False), fit(d, which, True); p1, p0 = f1.x, f0.x
            rec.update({f"{which}_dAIC": 2 * (f0.fun - f1.fun) - 4, f"{which}_LR": 2 * (f0.fun - f1.fun), f"{which}_dlo": p1[5], f"{which}_dhi": p1[6], f"{which}_beta": p1[1], f"{which}_gamma": p1[2], f"{which}_beta0": p0[1], f"{which}_gamma0": p0[2]})
            lo_rb, hi_rb = lapses(p1, +1); lo_lb, hi_lb = lapses(p1, -1)
            rec.update({f"{which}_lo_rightblock": lo_rb, f"{which}_lo_leftblock": lo_lb, f"{which}_hi_rightblock": hi_rb, f"{which}_hi_leftblock": hi_lb})
        rows.append(rec)
        if (k + 1) % 5 == 0: print(f"  {label}: {k + 1} mice | {time.time() - t0:.0f} s", flush=True)
    return pd.DataFrame(rows)
def med_ci(v, n=2000):
    v = np.asarray(v, float); bs = [np.median(v[rb.integers(0, len(v), len(v))]) for _ in range(n)]; return np.median(v), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
def report(F, label):
    say(f"\n[D2-2] {label}: {len(F)} mice")
    for which, nm in (("A", "A: prior = block log-odds"), ("B", "B: prior = recent-10 log-odds")):
        m, lo, hi = med_ci(F[f"{which}_dAIC"]); n_pos = int((F[f"{which}_dAIC"] > 0).sum())
        say(f"  {nm}: block-dependent lapses preferred (dAIC > 0, 2 extra params) in {n_pos}/{len(F)} mice; median dAIC {m:+.1f} [{lo:+.1f}, {hi:+.1f}]; total dAIC {F[f'{which}_dAIC'].sum():+.0f}")
        lr = F[f"{which}_LR"].sum(); say(f"     pooled likelihood-ratio test over mice (2 extra parameters per mouse): LR = {lr:.0f} on {2 * len(F)} df, p = {stats.chi2.sf(lr, 2 * len(F)):.2g}")
        for nmk, key in (("d_lo (error rate on LEFT stimuli vs block; expected > 0)", "dlo"), ("d_hi (error rate on RIGHT stimuli vs block; expected < 0)", "dhi")):
            m, lo, hi = med_ci(F[f"{which}_{key}"]); exp = (F[f"{which}_{key}"] > 0) if key == "dlo" else (F[f"{which}_{key}"] < 0)
            say(f"     {nmk}: median {m:+.2f} [{lo:+.2f}, {hi:+.2f}]; expected sign in {int(exp.sum())}/{len(F)} mice (sign test p = {stats.binomtest(int(exp.sum()), len(F), .5).pvalue:.4f})")
        eLo = F[f"{which}_lo_rightblock"] - F[f"{which}_lo_leftblock"]; eHi = F[f"{which}_hi_rightblock"] - F[f"{which}_hi_leftblock"]
        m1, a1, b1 = med_ci(eLo); m2, a2, b2 = med_ci(eHi)
        say(f"     error rate on LEFT stimuli, right block minus left block (expected +): {m1 * 100:+.1f} points [{a1 * 100:+.1f}, {b1 * 100:+.1f}]; error rate on RIGHT stimuli, right block minus left block (expected -): {m2 * 100:+.1f} points [{m2 * 0 + a2 * 100:+.1f}, {b2 * 100:+.1f}]")
        sh0 = 2.7726 * F[f"{which}_gamma0"] / F[f"{which}_beta0"]; sh1 = 2.7726 * F[f"{which}_gamma"] / F[f"{which}_beta"]
        ma, la, ha = med_ci(sh0 * 100); mb, lb, hb = med_ci(sh1 * 100)
        say(f"     boundary shift (0.2 vs 0.8) in % contrast: constant lapses {ma:.1f} [{la:.1f}, {ha:.1f}] -> block-dependent lapses {mb:.1f} [{lb:.1f}, {hb:.1f}]")

In [ ]:
# ===================== fit =====================
allm = sorted(B.subject.unique()); FA = run(allm, "all mice"); FA.to_csv(f"{OUT}/lapse_fits_all.csv", index=False); report(FA, "all mice")
good_sessions = set(acc1[acc1 >= 0.8].index); Bg = B[B.eid.isin(good_sessions)]; cg = Bg.groupby("subject").size(); mg = sorted(cg[cg >= 600].index)
if len(mg) >= 8:
    Bsave = B; B = Bg; FG = run(mg, "clean sessions (accuracy >= 0.8 at 100% contrast)"); B = Bsave; FG.to_csv(f"{OUT}/lapse_fits_clean.csv", index=False); report(FG, f"robustness: only sessions with accuracy >= 0.8 at 100% contrast")

In [ ]:
# ===================== plot =====================
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for j, which in enumerate(("A", "B")):
    ax[j].scatter(FA[f"{which}_dlo"], FA[f"{which}_dhi"], c=np.where(FA[f"{which}_dAIC"] > 0, "C3", "C7"), s=30); ax[j].axhline(0, c="gray", lw=.5); ax[j].axvline(0, c="gray", lw=.5)
    ax[j].set_xlabel("d_lo (expected > 0)"); ax[j].set_ylabel("d_hi (expected < 0)"); ax[j].set_title(f"{which}: lapse slopes per mouse (red = block-dependent lapses preferred)")
g = per_mouse_gap(1.0); ax[2].hist(g.values, bins=15); ax[2].axvline(0, c="k"); ax[2].set_xlabel("accuracy gap at 100% contrast (congruent - conflict)"); ax[2].set_title("per mouse")
plt.tight_layout(); plt.savefig(f"{OUT}/lapse_overview.png", dpi=130); plt.close()

In [ ]:
# ===================== LLaVA analogy (optional: needs full_rows_all.csv from the main LLaVA run, attach that dataset too) =====================
lp = glob.glob("/kaggle/input/**/full_rows_all.csv", recursive=True) + glob.glob("full_rows_all.csv")
if not lp: print("full_rows_all.csv not attached: skipping the LLaVA part")
else:
    d = pd.read_csv(lp[0]); h = d[d.order == "hist"].set_index("t"); n = d[d.order == "none"].set_index("t")
    M = h[["side", "contrast", "p_right", "n_hist", "n_hist_right", "delta"]].rename(columns={"delta": "d_hist"}); M["d_none"] = n.delta
    c = M[M.contrast == 1.0].copy(); tg = np.where(c.side == "right", 1, -1); c["m_none"], c["m_hist"] = c.d_none * tg, c.d_hist * tg          # margin toward the true side (positive = correct)
    say("\n[D2-3] LLaVA at 100% contrast (118 trials): margin toward the true side, delta times side sign (> 0 = correct)")
    def q(v): return f"n={len(v):3d} mean {v.mean():+.2f} | min {v.min():+.2f} | quartiles {np.percentile(v, 25):+.2f} / {np.percentile(v, 50):+.2f} / {np.percentile(v, 75):+.2f} | errors {int((v <= 0).sum())}"
    say("  no history:                       " + q(c.m_none))
    bl = c[c.p_right.isin([.2, .8])]; cf = ((bl.p_right == .8) & (bl.side == "left")) | ((bl.p_right == .2) & (bl.side == "right"))
    say("  history, block-congruent trials:  " + q(bl[~cf].m_hist)); say("  history, block-conflict trials:   " + q(bl[cf].m_hist))
    f = c[c.n_hist == 10]; ext = (f.n_hist_right >= 8) | (f.n_hist_right <= 2); cf2 = ((f.n_hist_right >= 8) & (f.side == "left")) | ((f.n_hist_right <= 2) & (f.side == "right"))
    say("  history, count-congruent (>=8 or <=2 of 10): " + q(f[ext & ~cf2].m_hist)); say("  history, count-conflict:                     " + q(f[ext & cf2].m_hist))
    say("  (lapse-like = a small cluster of trials answering by the prior while the rest are untouched; compression-like = the whole distribution shifts toward 0)")

In [ ]:
open(f"{OUT}/D2_summary.txt", "w").write("\n".join(SUMMARY))
import zipfile
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUT): zf.write(f"{OUT}/{fn}", fn)
print("zip written:", f"{OUT}.zip")